# Hierarchical Clustering

In [ ]:
# Remember: library imports are ALWAYS at the top of the script, no exceptions!
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from math import ceil

## New imports
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram


sns.set()

In [ ]:
## Un-comment these if you want to use ydata_profiling

# !pip install -U ydata-profiling
# from ydata_profiling import ProfileReport


## Context
The data we will be using through the pratical classes comes from a small relational database whose schema can be seen below:
![Schema](https://raw.githubusercontent.com/fpontejos/DMDM_2223/main/figures/schema.png "Relation database schema")

## Reading the Data

In [ ]:
## Allow Colab to see Google Drive files

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Load csv file into a dataframe
## Paste the path here
data_path = "/content/drive/MyDrive/Colab Data/datamining.csv"

df = pd.read_csv(data_path)

In [ ]:
## Load csv file into a dataframe
## Alternative location of the csv file

## data_path = "https://raw.githubusercontent.com/fpontejos/DM1_2324/main/data/datamining.csv"

## df = pd.read_csv(data_path)

### Make a copy of your original dataset

why?

In [ ]:
df_original = df.copy()

### Metadata
- *id* - The unique identifier of the customer
- *age* - The year of birht of the customer
- *income* - The income of the customer
- *frq* - Frequency: number of purchases made by the customer
- *rcn* - Recency: number of days since last customer purchase
- *mnt* - Monetary: amount of € spent by the customer in purchases
- *clothes* - Number of clothes items purchased by the customer
- *kitchen* - Number of kitchen items purchased by the customer
- *small_appliances* - Number of small_appliances items purchased by the customer
- *toys* - Number of toys items purchased by the customer
- *house_keeping* - Number of house_keeping items purchased by the customer
- *dependents* - Binary. Whether or not the customer has dependents
- *per_net_purchase* - Percentage of purchases made online
- *education* - Education level of the customer
- *status* - Marital status of the customer
- *gender* - Gender of the customer
- *description* - Last customer's recommendation description

## Preprocess the Data

Remember what we did in the previous session

In [ ]:
# Sometimes it is not obvious that a value is missing
# For example if the value is an empty string

# replace "" by nans
df.replace("", np.nan, inplace=True)

In [ ]:
df["dependents"] = df["dependents"].astype("boolean")

In [ ]:
# Define metric and non-metric features. Why?
non_metric_features = ["education", "status", "gender", "dependents", "description"]

## This is saying that the metric features are all the other features that are not non-metric
## Need to be careful in case not all columns are to be used as features

# metric_features = df.columns.drop(non_metric_features).to_list()

## Or you can also specify manually
metric_features = ['age',
 'income',
 'frq',
 'rcn',
 'mnt',
 'clothes',
 'kitchen',
 'small_appliances',
 'toys',
 'house_keeping',
 'per_net_purchase']

### Fill missing values (Data imputation)

How can we fill missing values?


#### Using measures of central tendency

In [ ]:
# Creating a copy to apply central tendency measures imputation
df_central = df.copy()

In [ ]:
# count of missing values
df_central.isna().sum()

In [ ]:
medians = df_central[metric_features].median()
medians

In [ ]:
modes = df_central[non_metric_features].mode().loc[0]
modes

In [ ]:
## Fill NaNs using medians and modes

df_central.fillna(medians, inplace=True)
df_central.fillna(modes, inplace=True)

In [ ]:
df_central.isna().sum()  # checking how many NaNs we still have

In [ ]:
df = df_central.copy()

### Outlier removal

In [ ]:
def remove_outliers(df, filters):

  df_2 = df[filters]
  print('Percentage of data kept after removing outliers:', 100*(np.round(df_2.shape[0] / df.shape[0], 4)))

  return df_2



In [ ]:
# This may vary from session to session, and is prone to varying interpretations.
# A simple example is provided below:

manual_filters = (
    (df['house_keeping']<=50)
    &
    (df['kitchen']<=40)
    &
    (df['toys']<=35)
    &
    (df['education']!='OldSchool')
)

df_1 = remove_outliers(df, manual_filters)

In [ ]:
# Get the manual filtering version
df = df_1.copy()

In [ ]:
# How can we avoid having as many extreme values in 'rcn'?
print((df['rcn']>100).value_counts())

rcn_t = df['rcn'].copy()
rcn_t.loc[rcn_t>100] = 100

df['rcn'] = rcn_t

#### What about non-metric features?

In [ ]:
# Let's also remove status=Whatever
df.loc[df['status'] == 'Whatever', 'status'] = df['status'].mode()[0]


### Feature Engineering and Feature Selection

In [ ]:
df['birth_year'] = df['age']
df['age'] = datetime.now().year - df['birth_year']

df['spent_online'] = (df['per_net_purchase'] / 100) * df['mnt']

#### Redundancy


In [ ]:
# Select variables according to their correlations
df.drop(columns=['birth_year', 'age', 'mnt'], inplace=True)

In [ ]:
# Updating metric_features
metric_features.append("spent_online")
metric_features.remove("mnt")
metric_features.remove("age")

In [ ]:
metric_features

#### Relevancy
Selecting variables based on the relevancy of each one to the task. Example: remove uncorrelated variables with the target, stepwise regression, use variables for product clustering, use variables for socio-demographic clustering, ...

Variables that aren't correlated with any other variable are often also not relevant. In this case we will not focus on this a lot since we don't have a defined task yet.

### Data Normalization

#### Standard Scaling

In [ ]:
df_standard = df.copy()

In [ ]:
scaler = StandardScaler()
scaled_feat = scaler.fit_transform(df_standard[metric_features])
df_standard[metric_features] = scaled_feat


In [ ]:
df = df_standard.copy()

### One-hot encoding

In [ ]:
df_ohc = df.copy()

In [ ]:
def get_ohc_df(df, feats):
  # Use OneHotEncoder to encode the categorical features.
  # Get feature names and create a DataFrame
  # with the one-hot encoded categorical features (pass feature names)

  ohc = OneHotEncoder(sparse_output=False, drop="first")
  ohc_feat = ohc.fit_transform(df[feats])
  ohc_feat_names = ohc.get_feature_names_out()
  ohc_df = pd.DataFrame(ohc_feat, index=df.index, columns=ohc_feat_names)

  # Reassigning df to contain ohc variables
  df_ohc = pd.concat([df, ohc_df], axis=1)

  ## Return the df with the one-hot encoded features
  ## Also return the OneHotEncoder model (ohc)
  return df_ohc, ohc

df_ohc, ohc = get_ohc_df(df, non_metric_features)


In [ ]:
oh_features = ohc.get_feature_names_out().tolist()
oh_features

In [ ]:
df_ohc.columns

In [ ]:
df = df_ohc.copy()

In [ ]:
df.isna().sum()

### OR... Import preprocessed data


## Hierarchical Clustering

What is hierarchical clustering? How does it work? How does it relate to the distance matrix?


## The distance matrix


![](https://raw.githubusercontent.com/fpontejos/DMDM_2223/main/figures/hc_distance_matrix.png)

https://dashee87.github.io/data%20science/general/Clustering-with-Scikit-with-GIFs/

![](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/hierarch.gif)


## Different types of linkage
![](https://scikit-learn.org/stable/_images/sphx_glr_plot_linkage_comparison_001.png)

## How are they computed?
![](https://raw.githubusercontent.com/fpontejos/DMDM_2223/main/figures/linkage_types.jpeg)

**Ward linkage**: minimizes the sum of squared differences within all clusters. It is a variance-minimizing approach and in this sense is similar to the k-means objective function but tackled with an agglomerative hierarchical approach.



### Characteristics:
- *bottom up approach*: each observation starts in its own cluster, and clusters are successively merged together
- *greedy/local algorithm*: at each iteration tries to minimize the distance of cluster merging
- *no realocation*: after an observation is assigned to a cluster, it can no longer change
- *deterministic*: you always get the same answer when you run it
- *scalability*: can become *very slow* for a large number of observations

### How to apply Hierarchical Clustering?
**Note: Which types of variables should be used for hierarchical clustering?**

In [ ]:
# Performing HC
hclust = AgglomerativeClustering(linkage='ward',
                                 metric='euclidean',
                                 n_clusters=5)
hc_labels = hclust.fit_predict(df[metric_features])
hc_labels


In [ ]:
def get_mean_bylabel(df, feats, label_name):
  # Characterizing the clusters
  return df[feats+[label_name]].groupby(label_name).mean()


In [ ]:
df_hc_labeled = pd.concat((df, pd.Series(hc_labels, name='hc_labels', index=df.index)),
                        axis=1)
df_hc_labeled

In [ ]:
## Characterizing the clusters
## We will talk more about characterizing the clusters in the next sessions

hc_means_df = get_mean_bylabel(df_hc_labeled, metric_features, 'hc_labels')
hc_means_df

In [ ]:
## Let's add some style
## More details here: https://pandas.pydata.org/docs/reference/api/pandas.io.formats.style.Styler.background_gradient.html

hc_means_df.style.background_gradient(axis=0)

## Extra: Try to find out how to use a different color scheme

**How do we know these are the best parameters to use?**

```python
AgglomerativeClustering(
  linkage='ward',
  metric='euclidean',
  n_clusters=5
)
```

#### Defining the number of clusters

#### Let's plot the R2

**We need to understand that:**
$$SS_{t} = SS_{w} + SS_{b}$$

---

$$SS_{t} = \sum\limits_{i = 1}^n {{{({x_i} - \overline x )}^2}}$$

$$SS_{w} = \sum\limits_{k = 1}^K {\sum\limits_{i = 1}^{{n_k}} {{{({x_i} - {{\overline x }_k})}^2}} }$$

$$SS_{b} = \sum\limits_{k = 1}^K {{n_k}{{({{\overline x }_k} - \overline x )}^2}}$$

, where $n$ is the total number of observations,

$x_i$ is the vector of the $i^{th}$ observation,

$\overline x$ is the centroid of the data,

$K$  is the number of clusters,

$n_k$ is the number of observations in the $k^{th}$ cluster and

$\overline x_k$ is the centroid of the $k^{th}$ cluster.


---


![](https://raw.githubusercontent.com/fpontejos/DM1_2324/main/figures/ssw_ssb.png)

In [ ]:
def get_r2_hc(df, link_method, max_nclus, min_nclus=1, dist="euclidean"):
    """This function computes the R2 for a set of cluster solutions given by the application of a hierarchical method.
    The R2 is a measure of the homogenity of a cluster solution. It is based on SSt = SSw + SSb and R2 = SSb/SSt.

    Parameters:
    df (DataFrame): Dataset to apply clustering
    link_method (str): either "ward", "complete", "average", "single"
    max_nclus (int): maximum number of clusters to compare the methods
    min_nclus (int): minimum number of clusters to compare the methods. Defaults to 1.
    dist (str): distance to use to compute the clustering solution. Must be a valid distance. Defaults to "euclidean".

    Returns:
    ndarray: R2 values for the range of cluster solutions
    """
    def get_ss(df):
        ss = np.sum(df.var() * (df.count() - 1))
        return ss  # return sum of sum of squares of each df variable

    sst = get_ss(df)  # get total sum of squares

    r2 = []  # where we will store the R2 metrics for each cluster solution

    for i in range(min_nclus, max_nclus+1):  # iterate over desired ncluster range
        cluster = AgglomerativeClustering(n_clusters=i, metric=dist, linkage=link_method)


        hclabels = cluster.fit_predict(df) #get cluster labels


        df_concat = pd.concat((df, pd.Series(hclabels, name='labels', index=df.index)), axis=1)  # concat df with labels


        ssw_labels = df_concat.groupby(by='labels').apply(get_ss)  # compute ssw for each cluster labels


        ssb = sst - np.sum(ssw_labels)  # remember: SST = SSW + SSB


        r2.append(ssb / sst)  # save the R2 of the given cluster solution

    return np.array(r2)

In [ ]:
## This function plots the R2 values for each linkage method for cluster sizes from 1 to 10

def plot_r2_linkage(df, max_nclus):
    # Prepare input
    hc_methods = ["ward", "complete", "average", "single"]
    # Call function defined above to obtain the R2 statistic for each hc_method
    max_nclus = 10
    r2_hc_methods = np.vstack(
        [
            get_r2_hc(df=df, link_method=link, max_nclus=max_nclus)
            for link in hc_methods
        ]
    ).T
    r2_hc_methods = pd.DataFrame(r2_hc_methods, index=range(1, max_nclus + 1), columns=hc_methods)

    sns.set()
    # Plot data
    fig = plt.figure(figsize=(11,5))
    sns.lineplot(data=r2_hc_methods, linewidth=2.5, markers=["o"]*4)

    # Finalize the plot
    fig.suptitle("R2 plot for various hierarchical methods", fontsize=21)
    plt.gca().invert_xaxis()  # invert x axis
    plt.legend(title="HC methods", title_fontsize=11)
    plt.xticks(range(1, max_nclus + 1))
    plt.xlabel("Number of clusters", fontsize=13)
    plt.ylabel("R2 metric", fontsize=13)

    plt.show()

In [ ]:
## This will take several minutes
max_nclus = 10
plot_r2_linkage(df[metric_features], max_nclus)

#
### Let's plot the dendrogram

Where is the first big jump in the dendrogram?


In [ ]:
def plot_dendrogram(df, feats,
                    linkage='ward', distance='euclidean',
                    y_threshold = 75):

  # setting distance_threshold=0 and n_clusters=None ensures we compute the full tree
  hclust = AgglomerativeClustering(linkage=linkage,
                                  metric=distance,
                                  distance_threshold=0,
                                  n_clusters=None)

  hclust.fit_predict(df[feats])


  # Adapted from:
  # https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html#sphx-glr-auto-examples-cluster-plot-agglomerative-dendrogram-py

  # create the counts of samples under each node (number of points being merged)
  counts = np.zeros(hclust.children_.shape[0])
  n_samples = len(hclust.labels_)

  # hclust.children_ contains the observation ids that are being merged together
  # At the i-th iteration, children[i][0] and children[i][1] are merged to form node n_samples + i
  for i, merge in enumerate(hclust.children_):
      # track the number of observations in the current cluster being formed
      current_count = 0
      for child_idx in merge:
          if child_idx < n_samples:
              # If this is True, then we are merging an observation
              current_count += 1  # leaf node
          else:
              # Otherwise, we are merging a previously formed cluster
              current_count += counts[child_idx - n_samples]
      counts[i] = current_count



  ## Create linkage matrix

  # the hclust.children_ is used to indicate the two points/clusters being merged (dendrogram's u-joins)
  # the hclust.distances_ indicates the distance between the two points/clusters (height of the u-joins)
  # the counts indicate the number of points being merged (dendrogram's x-axis)
  linkage_matrix = np.column_stack(
      [hclust.children_, hclust.distances_, counts]
  ).astype(float)






  # Plot the corresponding dendrogram

  sns.set()
  fig = plt.figure(figsize=(11,5))


  # The Dendrogram parameters need to be tuned
  # "dendrogram" function will plot our dendrogram
  dendrogram(linkage_matrix,
            truncate_mode='level',
            p=5,
            color_threshold=y_threshold,
            above_threshold_color='k')




  plt.hlines(y_threshold, 0, 1000, colors="r", linestyles="dashed")
  plt.title(f'Hierarchical Clustering - {linkage.title()}\'s Dendrogram', fontsize=21)
  plt.xlabel('Number of points in node (or index of point if no parenthesis)')
  plt.ylabel(f'{distance.title()} Distance', fontsize=13)
  plt.show()



In [ ]:
plot_dendrogram(df, metric_features, y_threshold=100)

## y_threshold is here for visual demonstration of the cutoff
## You specify this value based on where you want the dashed red line to be drawn
## Try to test different values for y_threshold

#### Final Hierarchical Clustering Solution

In [ ]:
# 4 cluster solution
linkage = 'ward'
distance = 'euclidean'
hc4 = AgglomerativeClustering(linkage=linkage, metric=distance, n_clusters=4)
hc4_labels = hc4.fit_predict(df[metric_features])

In [ ]:
df_hc4 = pd.concat((df, pd.Series(hc4_labels,
                                  name='hc4_labels',
                                  index=df.index)),
                        axis=1)
df_hc4

In [ ]:
# 5 cluster solution
linkage = 'ward'
distance = 'euclidean'
hc5 = AgglomerativeClustering(linkage=linkage, metric=distance, n_clusters=5)
hc5_labels = hc5.fit_predict(df[metric_features])

In [ ]:
## Let's put both HC4 and HC5 labels in the same df to compare

df_hc4_hc5 = pd.concat((df,
                        pd.Series(hc4_labels,
                                  name='hc4_labels',
                                  index=df.index),
                        pd.Series(hc5_labels,
                                  name='hc5_labels',
                                  index=df.index)
                        ), axis=1)

In [ ]:
df_hc4_hc5

In [ ]:
pd.crosstab(df_hc4_hc5['hc5_labels'],
           df_hc4_hc5['hc4_labels'])

In [ ]:
## Which final clustering?

## Next: K-Means Clustering

### Questions?

## [OPTIONAL] Exercise

To practice your clustering skills, you can do the same exercises on a different dataset.


The Spaceship Titanic Dataset has been loaded for you in the cells below. You can find more information about this dataset from the Kaggle link.


Using this notebook as a guide, try to answer the questions that follow.


---

Addison Howard, Ashley Chow, Ryan Holbrook. (2022). Spaceship Titanic. Kaggle. https://kaggle.com/competitions/spaceship-titanic


In [ ]:
titanic_df = pd.read_csv("https://raw.githubusercontent.com/fpontejos/DM1_2324/main/data/spaceship_titanic_dataset.csv")


In [ ]:
## Keep only the useful features that you identified in the previous exercise

titanic_metric_features = []
titanic_non_metric_features = []


In [ ]:
## Don't forget to do the preprocessing steps you performed in the previous exercise


In [ ]:
## Perform the Hierarchical Clustering algorithm on your data
## making sure to use the appropriate hyperparameters such as number of clusters
## and linkage method